In [1]:
url = 'https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/0899f4de-4ac7-48d3-b317-a0365c88cfa8'
import requests
import pathlib
import zipfile
try:
  import wget
except:
  !pip install wget
  import wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=371b9535eb9526e65057e120e4c892057bb64b104c63fba246da8b8916407b2b
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
parent_path = pathlib.Path("data/")
zip_file = parent_path / "data.zip"
images_path = parent_path / "images"
if images_path.exists():
  print("Data is already present.")
else:
  images_path.mkdir(exist_ok=True, parents=True)
  if zip_file.exists():
    print("File already downloaded.")
  else:
    print('Downloading file...')
    wget.download(url, str(zip_file))
    with zipfile.ZipFile(zip_file) as z:
      print('Extracting file...')
      z.extractall(parent_path)
      print("Data fetched successfully.")

Extracting file...
Data fetched successfully.


In [3]:
!pip install split-folders
from splitfolders import ratio

In [4]:
data_path = parent_path / "SignAlphaSet/"
ratio(input=data_path, output=images_path, seed=42, ratio=(0.8, 0.1, 0.1))
test_path = images_path / "test/"
train_path = images_path / "train/"
validation_path = images_path / 'val/'

Copying files: 26000 files [00:09, 2794.22 files/s]


In [5]:
from torchvision.transforms import Compose, Resize, Normalize, ToTensor
auto_transform = Compose([Resize((224, 224)),  ToTensor(),Normalize(mean=(0,0,0), std=(1,1,1)),])
auto_transform

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0, 0, 0), std=(1, 1, 1))
)

In [6]:
from torchvision.datasets import ImageFolder
from torch.utils.data.dataloader import DataLoader
train_data = ImageFolder(train_path, auto_transform)
test_data = ImageFolder(test_path, auto_transform)
val_data = ImageFolder(validation_path, auto_transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
train_loader

In [7]:
try:
  from torchmetrics import Accuracy
except:
  !pip install torchmetrics
  from torchmetrics import Accuracy
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
acc_fn = Accuracy(num_classes=26, task='multiclass').to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.4 MB/s eta 0:00:00


In [58]:
class CNN(torch.nn.Module):
  def __init__(self):
    super().__init__()
    # self.features = torch.torch.nn.Sequential(
    #         torch.nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
    #         torch.nn.BatchNorm2d(64),
    #         torch.nn.ReLU(inplace=True),
    #         torch.nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
    #         torch.nn.BatchNorm2d(64),
    #         torch.nn.ReLU(inplace=True),
    #         torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    #         torch.nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
    #         torch.nn.BatchNorm2d(128),
    #         torch.nn.ReLU(inplace=True),
    #         torch.nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
    #         torch.nn.BatchNorm2d(128),
    #         torch.nn.ReLU(inplace=True),
    #         torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    #         torch.nn.Flatten()

    #     )

    # self.features = torch.nn.Sequential(
    #     torch.nn.Conv2d(3, 12, 3), # 224
    #     torch.nn.Conv2d(12, 10, 3),
    #     torch.nn.ReLU(),
    #     torch.nn.MaxPool2d((2, 2)), # 112
    #     torch.nn.Conv2d(10, 10, 3),
    #     torch.nn.ReLU(),
    #     torch.nn.Conv2d(10, 10, 3),
    #     torch.nn.ReLU(),
    #     torch.nn.MaxPool2d((4,4)), # 26
    #     torch.nn.Flatten()
    # )
    self.features = torch.nn.Sequential(
        torch.nn.Conv2d(3, 12, 3), # 224
        torch.nn.Conv2d(12, 10, 3),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d((2, 2)), # 112
        torch.nn.Conv2d(10, 10, 3),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d((4,4)), # 26
        torch.nn.Flatten()
    )
    self.classifier = torch.nn.Sequential(torch.nn.Dropout(p=0.3, inplace=True), torch.nn.Linear(in_features=7290, out_features=26))
  def forward(self, X):
    return self.classifier(self.features(X))

In [59]:
import torchvision

model = CNN()
model

CNN(
  (features): Sequential(
    (0): Conv2d(3, 12, kernel_size=(3, 3), stride=(1, 1))
    (1): Conv2d(12, 10, kernel_size=(3, 3), stride=(1, 1))
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): MaxPool2d(kernel_size=(4, 4), stride=(4, 4), padding=0, dilation=1, ceil_mode=False)
    (7): Flatten(start_dim=1, end_dim=-1)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=True)
    (1): Linear(in_features=7290, out_features=26, bias=True)
  )
)

In [60]:
from torchsummary import summary
summary(model=model.to(device), input_size=(3, 224, 224), batch_size=32)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [32, 12, 222, 222]             336
            Conv2d-2         [32, 10, 220, 220]           1,090
              ReLU-3         [32, 10, 220, 220]               0
         MaxPool2d-4         [32, 10, 110, 110]               0
            Conv2d-5         [32, 10, 108, 108]             910
              ReLU-6         [32, 10, 108, 108]               0
         MaxPool2d-7           [32, 10, 27, 27]               0
           Flatten-8                 [32, 7290]               0
           Dropout-9                 [32, 7290]               0
           Linear-10                   [32, 26]         189,566
Total params: 191,902
Trainable params: 191,902
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 18.38
Forward/backward pass size (MB): 472.55
Params size (MB): 0.73
Estimate

In [61]:
from tqdm import tqdm
def train_step(model, dataloader, optimiser, loss_fn, acc_fn, device):
  train_loss = 0
  train_accuracy = 0
  model.train()
  for images, labels in tqdm(dataloader, desc='Training'):
    images, labels = images.to(device), labels.to(device)
    y_probs = torch.softmax(model(images), dim=1)
    y_pred = torch.argmax(y_probs, dim=1)
    loss = loss_fn(y_probs, labels)
    acc = acc_fn(y_pred, labels)

    train_loss += loss.cpu().item()
    train_accuracy += acc.cpu().item()

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
  return train_loss / len(dataloader), train_accuracy / len(dataloader)


def evaluation_step(model, dataloader, loss_fn, acc_fn, device):
  val_loss = 0
  val_accuracy = 0
  model.eval()
  with torch.inference_mode():
    for images, labels in tqdm(dataloader, desc='Validating'):
      images, labels = images.to(device), labels.to(device)
      y_probs = torch.softmax(model(images), dim=1)
      y_pred = torch.argmax(y_probs, dim=1)
      loss = loss_fn(y_probs, labels)
      acc = acc_fn(y_pred, labels)

      val_loss += loss.cpu().item()
      val_accuracy += acc.cpu().item()

  return val_loss / len(dataloader), val_accuracy / len(dataloader)

In [64]:
class EarlyStopping:
  def __init__(self, patience=5, tolerance=0.00001):
    super().__init__()
    self.patience = patience
    self.counter = 0
    self.best_loss = None
    self.tolerance = tolerance

  def stop(self, loss):
    if self.best_loss is None:
      self.best_loss = loss
      return False
    else:
      if loss - self.tolerance >= self.best_loss:
        if self.counter >=self.patience:
          print("Early Stopping.")
          return True
        else:
          self.counter += 1
          return False
      return False

In [65]:
def train_model(model, train_loader, val_loader, optimiser, loss_fn, acc_fn, device, epochs):
  train_losses = []
  val_losses = []
  val_accs = []
  train_accs = []
  early_stopper = EarlyStopping(2)
  for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}/{epochs}")
    train_loss, train_acc = train_step(model=model, dataloader=train_loader, optimiser=optimiser, loss_fn=loss_fn, acc_fn=acc_fn, device=device)
    val_loss, val_acc = evaluation_step(model=model, dataloader=val_loader, loss_fn=loss_fn, acc_fn=acc_fn, device=device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    print(f"Train Accuracy: {train_acc} | Validation Accuracy: {val_acc} | Training Loss: {train_loss} | Validation Loss: {val_loss}")
    if early_stopper.stop(val_loss):
      break
  return {"train_losses": train_losses, "val_losses": val_losses, "train_accuracies":train_accs, "val_accuracies":val_accs}

In [66]:
%%time
loss_fn = torch.nn.CrossEntropyLoss()
model=model.to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

# train for 15 epochs
torch.manual_seed(42)
torch.cuda.manual_seed(42)
train_loss, val_loss, train_acc, val_acc = train_model(model=model, train_loader=train_loader, val_loader=val_loader, acc_fn=acc_fn, loss_fn=loss_fn, device=device, optimiser=optimiser, epochs=15)

Epoch: 1/15


Validating: 100%|██████████| 41/41 [00:08<00:00,  4.72it/s]


Train Accuracy: 0.5166346153846154 | Validation Accuracy: 0.5701219512195121 | Training Loss: 2.808241294714121 | Validation Loss: 2.7488974885242743
Epoch: 2/15


Validating: 100%|██████████| 41/41 [00:09<00:00,  4.44it/s]


Train Accuracy: 0.615625 | Validation Accuracy: 0.6844512195121951 | Training Loss: 2.7048779876415545 | Validation Loss: 2.6374085414700392
Epoch: 3/15


...

Train Accuracy: 0.9144230769230769 | Validation Accuracy: 0.9127286585365854 | Training Loss: 2.407064963854276 | Validation Loss: 2.4085239491811614
Epoch: 14/15


Validating: 100%|██████████| 41/41 [00:09<00:00,  4.36it/s]


Train Accuracy: 0.9147596153846154 | Validation Accuracy: 0.9138719512195121 | Training Loss: 2.406414916698749 | Validation Loss: 2.4072389835264625
Epoch: 15/15


Validating: 100%|██████████| 41/41 [00:09<00:00,  4.12it/s]

Train Accuracy: 0.9150961538461538 | Validation Accuracy: 0.9146341463414634 | Training Loss: 2.405916286615225 | Validation Loss: 2.406204840032066
CPU times: user 18min 49s, sys: 2min 3s, total: 20min 52s
Wall time: 21min 13s


In [67]:
test_loss, test_accuracy = evaluation_step(model=model, loss_fn=loss_fn, dataloader=test_loader, acc_fn=acc_fn, device=device)
print(f"Test loss is {test_loss} & Accuracy is {test_accuracy}")

Validating: 100%|██████████| 41/41 [00:09<00:00,  4.43it/s]

Test loss is 2.40074597916952 & Accuracy is 0.9199695121951219


In [68]:
print('Saving model..')
torch.save(model.state_dict(), "CUSTOM_ASL_CNN_FROM_SCRATCH1.pth")
print("Saved")

Saving model..
Saved
